In [1]:
#| default_exp litert

# A fast decoder for Android

> The decoder through Google's LiteRT onto a Pixel 10a: five rewrites the converter and the GPU need, each tested.

Android has no Core ML. Its route to the GPU is LiteRT (formerly TensorFlow Lite) and its GPU delegate. This notebook
converts notebook 103's `Body` to LiteRT and runs it on a Pixel 10a.

It needs the phone plugged in with USB debugging on, so `nbdev-test` skips it. It also needs `uv`: Google's
converter, `litert-torch`, pins an older `typing-extensions` than Jupyter runs with, so it runs in a throwaway
environment of its own.

In [2]:
#| export
import re
import shutil
import subprocess
import types
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from kokoro.istftnet import AdaIN1d, UpSample1d

from text_to_speech.core import DATA
from text_to_speech.coreml import Body, HarmonicSource, exportable_decoder

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.15.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.8.0 is the most recent version that has been tested.


In [3]:
import pandas as pd
from fastcore.test import test_close

from text_to_speech.voice import analyse

## Converting

`to_tflite` saves a module with `torch.export`, converts it in its own environment (`uv run --no-project`, built once
and cached by uv), and runs the result on the Mac to check it against PyTorch. Like Core ML's, LiteRT models have
fixed shapes.

In [4]:
#| export
PHONE = DATA / "phone"
CONVERT = """
import sys, numpy as np, torch, litert_torch
pt2, io, out = sys.argv[1:]
io = np.load(io)
args = tuple(torch.tensor(io[f"arr_{i}"]) for i in range(len(io.files) - 1))
model = litert_torch.convert(torch.export.load(pt2).module(), args)
model.export(out)
print(np.abs(np.asarray(model(*args)).reshape(io["out"].shape) - io["out"]).max())
"""


@torch.no_grad()
def to_tflite(module, args, name):
    "Convert `module` at the shapes of `args` to PHONE/name.tflite. Returns its largest error against PyTorch."
    PHONE.mkdir(parents=True, exist_ok=True)
    module = module.eval()
    torch.export.save(torch.export.export(module, args), PHONE / f"{name}.pt2")
    np.savez(PHONE / f"{name}.npz", *[a.numpy() for a in args], out=module(*args).numpy())
    uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
    env = [uv, "run", "-q", "--no-project", "--python", "3.13", "--with", "torch==2.14.0", "--with", "litert-torch"]
    files = [PHONE / f"{name}.pt2", PHONE / f"{name}.npz", PHONE / f"{name}.tflite"]
    r = subprocess.run([*env, "python", "-c", CONVERT, *files], capture_output=True, text=True, check=True)
    return float(r.stdout.strip().splitlines()[-1])

The body for 128 frames (3.2 s), on a real sentence's inputs:

In [5]:
T = 128
u = analyse("Speed comes from the voice itself, not from stretching the audio afterwards.")
dec = exportable_decoder()
with torch.no_grad():
    args = (u.asr[..., :T], u.F0[:, : 2 * T], u.N[:, : 2 * T], u.style, HarmonicSource(dec)(u.F0[:, : 2 * T]))
    args = tuple(a.contiguous() for a in args)
    reference = Body(dec)(*args)
print(
    f"largest error {to_tflite(Body(dec), args, 'first_try'):.3f} on audio of RMS {reference.pow(2).mean().sqrt():.3f}"
)

/Users/henrydashwood/text-to-speech/.venv/lib/python3.14/site-packages/torch/export/pt2_archive/_package.py:415: UserWarning: No complete tensor found in the group! Returning the first one. This may cause issues when your weights are not on CPU.
  complete_tensor = get_complete_tensor(group, weights_dict)


largest error 0.497 on audio of RMS 0.048


Ten times the size of the signal. Converting the body cut off after each stage in turn finds where it first goes
wrong, and the culprit inside: a transposed convolution with padding.

In [6]:
x = torch.randn(1, 64, 100)
for name, layer in [
    ("kernel 3, stride 2, padding 1", torch.nn.ConvTranspose1d(64, 64, 3, stride=2, padding=1)),
    ("kernel 20, stride 10, padding 5", torch.nn.ConvTranspose1d(64, 32, 20, stride=10, padding=5)),
]:
    print(f"ConvTranspose1d, {name}: largest error {to_tflite(layer, (x,), 'layer'):.1e}")

ConvTranspose1d, kernel 3, stride 2, padding 1: largest error 3.8e+00
ConvTranspose1d, kernel 20, stride 10, padding 5: largest error 3.6e-07


One shape converts wrongly and the generator's happens not to. So: no padded transposed convolutions. That, and four
more things the converter or the GPU delegate can't take, each rewritten as the same maths.

### 1. Padded transposed convolutions

A transposed convolution's padding just crops its output. So convolve with no padding, then crop.

In [7]:
#| export
class CroppedConvTranspose1d(torch.nn.Module):
    "A padded ConvTranspose1d as an unpadded one and a crop."

    def __init__(self, ct):
        super().__init__()
        self.ct, self.crop = ct, (ct.padding[0], ct.padding[0] - ct.output_padding[0])
        ct.padding, ct.output_padding = (0,), (0,)

    def forward(self, x):
        y = self.ct(x)
        return y[..., self.crop[0] : y.shape[-1] - self.crop[1]]

In [8]:
layer = torch.nn.ConvTranspose1d(64, 64, 3, stride=2, padding=1, output_padding=1)
expected = layer(x)
rewritten = CroppedConvTranspose1d(layer)
test_close(rewritten(x), expected, eps=1e-5)
print(f"converted: largest error {to_tflite(rewritten, (x,), 'layer'):.1e}")

converted: largest error 6.0e-07


### 2. A depthwise transposed convolution

The decoder's last upsampling block has a transposed convolution with 1,090 channels, each its own group. The
converter turns it into 1,090 separate operations. A stride-2 transposed convolution is the same as putting a zero
between every pair of samples and doing an ordinary convolution with the kernel flipped, and an ordinary depthwise
convolution is one operation.

In [9]:
#| export
class ZeroStuffedConv1d(torch.nn.Module):
    "A depthwise ConvTranspose1d as zeros between samples, then a depthwise Conv1d with the kernel flipped."

    def __init__(self, ct):
        super().__init__()
        C, k, (s,), (p,), (op,) = ct.in_channels, ct.kernel_size[0], ct.stride, ct.padding, ct.output_padding
        self.conv = torch.nn.Conv1d(C, C, k, groups=C, bias=ct.bias is not None)
        self.conv.weight.data = ct.weight.data.flip(-1)
        if ct.bias is not None:
            self.conv.bias.data = ct.bias.data
        self.stride, self.pad = s, (k - 1 - p, k - 1 - p + op)

    def forward(self, x):
        B, C, L = x.shape
        z = torch.stack([x, *[torch.zeros_like(x)] * (self.stride - 1)], -1).reshape(B, C, L * self.stride)
        return self.conv(F.pad(z[..., : (L - 1) * self.stride + 1], self.pad))

In [10]:
layer = torch.nn.ConvTranspose1d(64, 64, 3, stride=2, groups=64, padding=1, output_padding=1)
test_close(ZeroStuffedConv1d(layer)(x), layer(x), eps=1e-5)

### 3. Gathers

Nearest-neighbour upsampling and a one-sample reflection pad both become `GATHER_ND`, which the GPU delegate doesn't
support. Doubling every sample is stacking a tensor with itself; reflecting one sample is copying the second sample to
the front.

In [11]:
#| export
class Nearest2(torch.nn.Module):
    "F.interpolate(x, scale_factor=2, mode='nearest'), as stacking and reshaping."

    def forward(self, x):
        return torch.stack([x, x], -1).reshape(x.shape[0], x.shape[1], 2 * x.shape[2])


class ReflectOne(torch.nn.Module):
    "ReflectionPad1d((1, 0)): the second sample, copied in front."

    def forward(self, x):
        return torch.cat([x[..., 1:2], x], -1)

In [12]:
test_close(Nearest2()(x), F.interpolate(x, scale_factor=2, mode="nearest"))
test_close(ReflectOne()(x), torch.nn.ReflectionPad1d((1, 0))(x))

### 4. Instance norm

Written out as means and a square root, inside Kokoro's AdaIN layer:

In [13]:
#| export
def _adain(self, x, s):
    "AdaIN1d with the instance norm written out as means and a square root."
    gamma, beta = torch.chunk(self.fc(s)[:, :, None], chunks=2, dim=1)
    mean = x.mean(-1, keepdim=True)
    var = ((x - mean) ** 2).mean(-1, keepdim=True)
    x = (x - mean) / torch.sqrt(var + self.norm.eps) * self.norm.weight[:, None] + self.norm.bias[:, None]
    return (1 + gamma) * x + beta

In [14]:
adain = AdaIN1d(128, 64).eval()
s = torch.randn(1, 128)
expected = adain(x, s)
test_close(_adain(adain, x, s), expected, eps=1e-5)

### 5. The inverse STFT's envelope

`ExactSTFT.inverse` divides by the sum of its overlapping windows, which it builds from a tensor of ones. The
converter folds that into a constant, and the GPU delegate refuses a constant as a transposed convolution's input. For
a fixed number of frames the envelope *is* a constant, so we precompute its reciprocal.

In [15]:
#| export
def _fixed_envelope(stft, frames):
    "For a fixed number of frames the inverse STFT's window envelope is a constant: precompute its reciprocal."
    pad = stft.n_fft // 2
    envelope = F.conv_transpose1d(torch.ones(1, 1, frames), stft.window_squared, stride=stft.hop_length)
    stft.register_buffer("inv_envelope", 1 / envelope[..., pad:-pad])

    def inverse(self, magnitude, phase):
        real = F.conv_transpose1d(magnitude * torch.cos(phase), self.weight_backward_real, stride=self.hop_length)
        imag = F.conv_transpose1d(magnitude * torch.sin(phase), self.weight_backward_imag, stride=self.hop_length)
        return (real - imag)[..., pad:-pad] * self.inv_envelope

    stft.inverse = types.MethodType(inverse, stft)

## Putting them together

`phone_decoder` makes the decoder for T frames with all five rewrites. It should be the same decoder:

In [16]:
#| export
def _rewrite(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.ConvTranspose1d) and child.groups > 1:
            setattr(module, name, ZeroStuffedConv1d(child))
        elif isinstance(child, torch.nn.ConvTranspose1d) and child.padding[0]:
            setattr(module, name, CroppedConvTranspose1d(child))
        elif isinstance(child, UpSample1d) and child.layer_type != "none":
            setattr(module, name, Nearest2())
        elif isinstance(child, torch.nn.ReflectionPad1d):
            setattr(module, name, ReflectOne())
        else:
            if isinstance(child, AdaIN1d):
                child.forward = types.MethodType(_adain, child)
            _rewrite(child)


def phone_decoder(T):
    "The decoder for T frames, rewritten for LiteRT and its GPU delegate: (HarmonicSource, Body)."
    dec = exportable_decoder()
    source = HarmonicSource(dec).eval()
    _rewrite(dec)
    _fixed_envelope(dec.generator.stft, T * 120 + 1)  # 120 source columns per frame
    return source, Body(dec).eval()

In [17]:
source, body = phone_decoder(T)
with torch.no_grad():
    test_close(body(*args), reference, eps=1e-4)
print(
    f"converted: largest error {to_tflite(body, args, f'body{T}'):.1e}, {(PHONE / f'body{T}.tflite').stat().st_size / 1e6:.0f} MB"
)

converted: largest error 7.0e-06, 214 MB


## On the phone

`adb` talks to the phone. Google's `benchmark_model` tool loads a LiteRT model on the phone and times it; we copy it
and the model to `/data/local/tmp`, the phone's folder for command-line tools.

In [18]:
#| export
ADB = shutil.which("adb") or str(Path.home() / "Library/Android/sdk/platform-tools/adb")
TOOL = "https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/android_aarch64_benchmark_model"
ON_PHONE = "/data/local/tmp"


def adb(*args):
    "Run an adb command. Returns what it printed, stdout and stderr together."
    r = subprocess.run([ADB, *args], capture_output=True, text=True, check=False)  # adb's exit code says little
    return r.stdout + r.stderr


def install(path):
    "Copy a file to the phone's tools folder, downloading the benchmark tool first if it's that."
    path = Path(path)
    if path.name == "benchmark_model" and not path.exists():
        urllib.request.urlretrieve(TOOL, path)
    adb("push", str(path), f"{ON_PHONE}/{path.name}")
    adb("shell", "chmod", "+x", f"{ON_PHONE}/{path.name}")


def benchmark(model, *flags, runs=10):
    "Time `model` on the phone: (seconds to load, seconds per inference, the tool's output)."
    out = adb(
        "shell",
        f"{ON_PHONE}/benchmark_model",
        f"--graph={ON_PHONE}/{model}",
        f"--num_runs={runs}",
        "--warmup_runs=2",
        *flags,
    )
    m = re.search(r"Init: ([\d.e+]+).*Inference \(avg\): ([\d.e+]+)", out)
    if not m:
        raise RuntimeError(out[-2000:])
    return float(m[1]) / 1e6, float(m[2]) / 1e6, out

In [19]:
print(adb("shell", "getprop", "ro.product.model").strip(), "with", adb("shell", "getprop", "ro.soc.model").strip())
install(PHONE / "benchmark_model")
install(PHONE / f"body{T}.tflite")
cache = [f"--delegate_serialize_dir={ON_PHONE}/gpu_cache", f"--delegate_serialize_token=body{T}"]
adb("shell", "mkdir", "-p", f"{ON_PHONE}/gpu_cache")
rows = []
for name, flags in [("CPU, 4 threads", ["--num_threads=4"]), ("GPU, 16-bit", ["--use_gpu=true", *cache])]:
    load, per_call, _ = benchmark(f"body{T}.tflite", *flags)
    rows.append({"where": name, "load (s)": load, "per call (s)": per_call, "x real time": T * 0.025 / per_call})
pd.DataFrame(rows).round(2)

Pixel 10a with Tensor G4


,where,load (s),per call (s),x real time
0,"CPU, 4 threads",0.33,2.2,1.45
1,"GPU, 16-bit",0.48,0.3,10.62


The GPU runs about seven times faster than the CPU, which only just keeps up. The GPU's first load compiles programs
for the model and takes a couple of minutes; with `--delegate_serialize_dir` it saves them, and later loads take well
under a second (as here, where an earlier run had already saved them). An app would do that once, at install.

## Summary

- Five rewrites, each the same maths, get the decoder through LiteRT's converter and onto the GPU.
- On a Pixel 10a's GPU, about 10x real time; on its CPU, about 1.4x.
- The phone GPU's own output hasn't been checked by ear: Google's tool can't save outputs, so that needs an app. The
  appendix has the full sweep, and where the CPU's time goes.